In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica direta (RandomForestRegressor) aprendida apenas com dados sem falha
+ Aplicação global ponto a ponto em toda a base
+ Classificação multiclasse com split por temperatura sem sobreposição
Autor: Luiz Eduardo Abdala José
"""

import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

# ========= PARÂMETROS =========
ARQ_BASE = "base-completo--.pkl"   # ✅ base completa
REF_TEMP = 20
FREQ_MIN_KHZ = 30
FREQ_MAX_KHZ = 70
SMOOTH_WIN = 5

RF_COMP_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0
)

RF_CLASSIF_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# ========= FUNÇÕES =========
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs)[order]

def add_extra_features(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1)-X.min(axis=1)).reshape(-1,1)
    return np.hstack([X, mu, sd, amp])

def add_temp_feature(X_aug, temp_vec):
    return np.hstack([X_aug, np.asarray(temp_vec).reshape(-1,1)])

def moving_average(arr, win):
    """
    Média móvel centralizada com saída do mesmo tamanho do vetor original.
    """
    if win <= 1 or win % 2 == 0:
        return arr.copy()

    pad = win // 2
    # padding nas bordas com valores replicados
    arr_pad = np.pad(arr, (pad, pad), mode='edge')
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode='valid')

    # garante mesmo comprimento que entrada
    if len(smooth) > len(arr):
        smooth = smooth[:len(arr)]
    elif len(smooth) < len(arr):
        smooth = np.pad(smooth, (0, len(arr) - len(smooth)), mode='edge')
    return smooth

# ========= ETAPA 1 – CARREGAMENTO =========
df = pd.read_pickle(ARQ_BASE)
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz_khz = fhz/1e3

df_sem = df[df["falha"]==0].copy()
print(f"Amostras sem falha: {len(df_sem)} | total: {len(df)}")

# ========= ETAPA 2 – REFERÊNCIA REAL =========
pool_20 = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
y_ref = np.median(pool_20, axis=0) if len(pool_20)>0 else np.median(df_sem[fcols], axis=0)
print(f"Referência calculada com {len(pool_20)} curvas @ {REF_TEMP}°C.")

# ========= ETAPA 3 – TREINO COMPENSAÇÃO =========
X_sem = df_sem[fcols].to_numpy(float)
T_sem = df_sem["temperatura_c"].to_numpy(float)
Y_target = (y_ref[None,:] - X_sem)
X_aug_sem = add_extra_features(X_sem)
X_comp_sem = add_temp_feature(X_aug_sem, T_sem)

print("\n🔹 Treinando modelo de compensação térmica...")
rf_comp = RandomForestRegressor(**RF_COMP_PARAMS)
rf_comp.fit(X_comp_sem, Y_target)
print("✅ RF-Comp treinado com sucesso.")

# ========= ETAPA 4 – APLICA COMPENSAÇÃO =========
X_all = df[fcols].to_numpy(float)
T_all = df["temperatura_c"].to_numpy(float)
X_aug_all = add_extra_features(X_all)
X_comp_all = add_temp_feature(X_aug_all, T_all)

print("🔹 Aplicando compensação térmica...")
Y_hat = X_all + rf_comp.predict(X_comp_all)

# Suaviza com média móvel
if SMOOTH_WIN > 1 and SMOOTH_WIN % 2 == 1:
    for i in range(Y_hat.shape[0]):
        Y_hat[i] = moving_average(Y_hat[i], SMOOTH_WIN)

df_comp = df.copy(); df_comp[fcols] = Y_hat
print("✅ Compensação aplicada com sucesso.")

# ========= ETAPA 5 – SPLIT SEM OVERLAP =========
temps_all = sorted(df_comp["temperatura_c"].unique())
temps_train = temps_all[::2]
temps_test  = temps_all[1::2]

print(f"Temperaturas treino: {temps_train}")
print(f"Temperaturas teste:  {temps_test}")

df_train = df_comp[df_comp["temperatura_c"].isin(temps_train)]
df_test  = df_comp[df_comp["temperatura_c"].isin(temps_test)]

X_train = df_train[fcols].to_numpy(float)
y_train = df_train["falha"].to_numpy(int)
X_test  = df_test[fcols].to_numpy(float)
y_test  = df_test["falha"].to_numpy(int)

print(f"Amostras treino: {len(X_train)} | teste: {len(X_test)}")

# ========= ETAPA 6 – CLASSIFICAÇÃO =========
print("\n🔹 Treinando classificador multiclasse...")
clf = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n== RESULTADOS RANDOM FOREST (RF Direto, split sem overlap) ==")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

# ========= ETAPA 7 – GRÁFICO DE EXEMPLO =========
print("\n🔹 Gerando gráfico de exemplo...")
idx_show = df_test.index[10] if len(df_test) > 10 else df_test.index[0]

plt.figure(figsize=(9,5))
plt.plot(fhz_khz, y_ref, '--', c='black', lw=1.2, label=f"Referência {REF_TEMP}°C (sem falha)")
plt.plot(fhz_khz, df.loc[idx_show, fcols], c='tab:red', alpha=0.6,
         label=f"Original {df.loc[idx_show,'temperatura_c']}°C (falha={df.loc[idx_show,'falha']})")
plt.plot(fhz_khz, df_comp.loc[idx_show, fcols], c='tab:blue', lw=2,
         label=f"Compensado {df.loc[idx_show,'temperatura_c']}°C")
plt.title(f"Compensação RF — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.legend()
plt.tight_layout()
plt.grid(alpha=0.3)
plt.show()
